# Local Judge Validation — can an open-weights grader replace the API oracle?  `[MEASUREMENT VALIDITY]`

Scores conversations **already on disk** with a locally-served open-weights model and checks it against the
two graders we already have (gpt-4o-mini and Claude Haiku 4.5). Costs **$0** in API spend — only GPU time.

Why this is worth doing before anything else: the oracle is both the training reward and the measurement
instrument, and API spend is the binding constraint on the whole project. If an open-weights judge reproduces
the contrasts, every future experiment is free. If it doesn't, we found that out for free instead of inside a
training run.

**Two gates before the full sweep**, because the failure modes are different from a hosted judge's:

1. **Schema** — does this backend honour each rubric's `json_schema`? `run_judge_scoring` swallows per-call
   errors and skips the conversation, so a rubric the model can't satisfy shows up as *biased missingness*,
   not as an error.
2. **Discrimination** — does it separate two arms the primary oracle puts far apart? A small model can honour
   the schema perfectly and still answer from a template (every item a 4). That parses, writes valid CSVs,
   and produces a judge that cannot tell any two arms apart — and nothing downstream would flag it.

`judge_plan.check_rubric_parity` is **not** the gate here: it asks a static question about constraints we
strip for Claude, and a local OpenAI-compatible server strips nothing.

Outputs land in `data/eval_scores/judge=<tag>/rep=0/` beside every other grader's; scoring is resume-safe
(existing CSVs are skipped). This is a scoring notebook like `Run_Eval.ipynb` — **not** part of `render_results.py`.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import asyncio
import numpy as np, pandas as pd
pd.set_option("display.width", 185, "display.max_columns", 50)
from eda_analysis.scoring import (registry as osc_config, conversations as osc_data,
                                  judge as jc, local_server as lsrv)
from eda_analysis import reliability as rel

# ╔═══ KNOBS ═══════════════════════════════════════════════════════════════════╗
# The grader under test. Verify the exact repo id on HF before running — the E2B/E4B suffixes
# belong to the Gemma 3n line, and gated repos need your HF token.
LOCAL_MODEL = "google/gemma-3n-E4B-it"
JUDGE_TAG   = None          # None -> "local_<shorttag>", e.g. local_gemma3nE4B. STABLE once scored.

# Anchor states: base + both endpoints + the GRPO peak. Persona pairing across arms is valid only
# at matched iterations (same seed+k+1 shuffle) — these are matched.
SUBSET_MODELS = ["PTOExp3_LA0_Base", "PTOExp3_LA0_I10", "GRPOExp3_LA0_I8", "GRPOExp3_LA0_I10"]
REFERENCE_MODEL = "PTOExp3_LA0_Base"     # the "gain over base" baseline for gain_retention
METRICS   = ["Q1", "Q2", "MICI"]         # Q1+Q2 = headline reward; MICI = the sycophancy claim
SUBSET_N  = None                          # None = all 96 convs per model
CONCURRENCY = 32                          # local server: raise until GPU util saturates

# vLLM. gpu_memory_utilization is a PRE-ALLOCATION, not a ceiling.
#   scoring on an idle GPU  -> HIGH (0.85): a bigger KV pool buys concurrency
#   sharing with a trainer  -> LOW (~0.25), and start the server FIRST
# On a 12 GB card a bf16 4B model plus a useful pool does NOT fit — quantize or use a bigger GPU.
GPU_MEM_UTIL  = 0.85
MAX_MODEL_LEN = 8192
SERVER_PORT   = 8000
EXTERNAL_BASE_URL = None    # set to e.g. "http://localhost:8000/v1" to use a server you started yourself

# ── $$ SWITCHES — nothing runs on open ──────────────────────────────────────────
RUN_GATES = False    # cheap: a few dozen calls
RUN_SWEEP = False    # the full subset sweep (len(SUBSET_MODELS) x METRICS x 96 calls)
# ╚═════════════════════════════════════════════════════════════════════════════╝
print(f"grader under test : {LOCAL_MODEL}")
print(f"anchor states     : {SUBSET_MODELS}")
print(f"metrics           : {METRICS}")
print(f"gates={RUN_GATES}  sweep={RUN_SWEEP}")

In [ ]:
# Colab only — install the server + mount Drive (the score lake and conversations live there).
# Uncomment on a fresh Colab runtime. Locally, use the repo .venv.
#
# !pip -q install vllm
# from google.colab import drive; drive.mount("/content/drive")
# import os; os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"   # persist weights across sessions
# from huggingface_hub import login; from google.colab import userdata
# login(token=userdata.get("huggingface"))                                # Gemma repos are gated

In [ ]:
# Conversations for the anchor states, straight from the auto-discovered registry.
exps = [e for e in osc_config.EXPERIMENTS if e.model_name in SUBSET_MODELS]
missing = set(SUBSET_MODELS) - {e.model_name for e in exps}
assert not missing, f"missing on disk: {missing} (Drive symlinks mounted?)"

names    = [e.model_name for e in exps]
layout   = osc_config.get_model_eval_layout(exps)
combined = osc_data.combine_data(osc_data.load_data(osc_config.resolve_paths(exps)), names)
print(combined.groupby("Model").size().rename("convs on disk"))

n_convs = SUBSET_N or int(combined.groupby("Model").size().min())
print(f"\nsweep would be {len(SUBSET_MODELS)} models x {len(METRICS)} metrics x {n_convs} convs "
      f"= {len(SUBSET_MODELS) * len(METRICS) * n_convs:,} calls  ($0 — local GPU)")

In [ ]:
# Start the server (or attach to one you started yourself via EXTERNAL_BASE_URL).
# Weight download on a cold Colab runtime can take a while; the log path is printed.
if EXTERNAL_BASE_URL:
    server = lsrv.ServerHandle(model=LOCAL_MODEL, base_url=EXTERNAL_BASE_URL)
    lsrv.wait_until_ready(server.base_url, timeout=120)
    print(f"attached to {server.base_url}")
else:
    server = lsrv.start_server(LOCAL_MODEL, port=SERVER_PORT,
                               gpu_memory_utilization=GPU_MEM_UTIL,
                               max_model_len=MAX_MODEL_LEN)

JUDGE = lsrv.local_judge(server.model, server.base_url, tag=JUDGE_TAG)
print("judge tag (score-lake partition):", JUDGE.tag)

## Gates — both must pass before the sweep

**Gate 1 (schema).** Every rubric must come back parseable, with the right number of item scores.
Any `n_fail > 0` means that metric would be silently under-sampled across the whole sweep.

**Gate 2 (discrimination).** `PTOExp3_LA0_Base` vs `PTOExp3_LA0_I10` is a gap the primary oracle
calls **+1.26 on Q1+Q2** — the largest, least ambiguous contrast in the experiment. A grader that
can't see *that* is not a measuring instrument, and `degenerate=True` (per-conversation SD ≈ 0)
means it is answering from a template regardless of what it read.

In [ ]:
if RUN_GATES:
    schema = await lsrv.probe_rubrics(JUDGE, combined, METRICS, n_convs=2)
    display(schema)
    assert (schema.n_fail == 0).all(), \
        "SCHEMA GATE FAILED — this backend cannot satisfy every rubric; those metrics would be " \
        "silently under-sampled. Check the guided-decoding backend or pick another model."

    disc = await lsrv.probe_discrimination(JUDGE, combined, METRICS,
                                           model_a=REFERENCE_MODEL, model_b="PTOExp3_LA0_I10",
                                           n_convs=12)
    display(disc)
    assert not disc.degenerate.any(), \
        "DISCRIMINATION GATE FAILED — near-zero variance across conversations: the grader is " \
        "answering from a template. Do not spend GPU-hours on the sweep."
    print("\nboth gates passed — the sweep is worth running.")
else:
    print("RUN_GATES=False — flip the knob in cell 1.")

## Sweep

Resume-safe: existing CSVs are skipped, so a killed Colab session is re-runnable at no cost.
Writes to `data/eval_scores/judge=<tag>/rep=0/` — its own partition, never another grader's.

In [ ]:
if RUN_SWEEP:
    stats = await jc.run_judge_scoring(JUDGE, combined, METRICS, layout,
                                       rep=0, concurrency=CONCURRENCY, subset_n=SUBSET_N)
    print(stats)
else:
    print("RUN_SWEEP=False — flip the knob in cell 1.")

## Does it agree with the graders we trust?

Same battery the second judge went through, so the numbers are directly comparable to
`notebooks/measurement/validity.ipynb` §2 — and the local judge is held to the same bar Haiku cleared.

⚠ **Never average this judge's raw scores with the others.** The level offset is model-dependent;
combine only contrasts or standardized quantities. What matters is whether **directions** survive.

In [ ]:
local_long = rel.load_judge_long(JUDGE.tag, reps=[0])
if local_long.empty:
    print("nothing scored yet for", JUDGE.tag, "— run the sweep first.")
else:
    prim_long = rel.load_primary_long(SUBSET_MODELS, METRICS, layout)
    display(rel.coverage_table(local_long, n_expected=n_convs))
    local_long, prim_long = rel.filter_complete_cells(local_long, prim_long, n_required=n_convs)

    print("\n── local judge vs PRIMARY (gpt-4o-mini) ──")
    display(rel.agreement(local_long, prim_long))

    pairs = rel.all_pairs_contrasts(local_long, prim_long, metrics=METRICS, models=SUBSET_MODELS)
    print("\n── sign preservation (the claim that actually matters) ──")
    display(rel.sign_preservation(pairs))

    print("\n── arm-mean variance: do they disagree about LEVEL or about ORDERING? ──")
    display(rel.variance_components_arm(local_long, prim_long, METRICS))

    print("\n── gain retention vs", REFERENCE_MODEL, "(the reward-hacking test) ──")
    display(rel.gain_retention(local_long, prim_long, reference_model=REFERENCE_MODEL,
                               metrics=METRICS))

In [ ]:
# Third leg: local vs the HELD-OUT judge (Haiku never played the patient, so this pair is the
# cleanest test of whether the local grader tracks a judge with no stake in the training reward).
HAIKU_TAG = "anthropic_claude-haiku-4-5"
haiku_long = rel.load_judge_long(HAIKU_TAG, reps=[0])
if local_long.empty or haiku_long.empty:
    print("need both the local sweep and the Haiku sweep on disk")
else:
    h = haiku_long[haiku_long.model.isin(SUBSET_MODELS) & haiku_long.metric.isin(METRICS)]
    l, h = rel.filter_complete_cells(local_long, h, n_required=n_convs)
    display(rel.agreement(l, h))
    display(rel.sign_preservation(rel.all_pairs_contrasts(l, h, metrics=METRICS,
                                                          models=SUBSET_MODELS)))

In [ ]:
# Free the GPU. Safe to call twice; a no-op for an EXTERNAL_BASE_URL server you own.
server.stop()